In [1]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [6]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name AS country, -- Use canonical country name from gdi
    AVG(CASE WHEN ud.month IN (7, 8, 9) THEN ud.uniques_estimate ELSE NULL END) AS previous_metric_avg,
    AVG(CASE WHEN ud.month IN (10, 11, 12) THEN ud.uniques_estimate ELSE NULL END) AS current_metric_avg
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd ON ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year = 2023
    AND ud.month IN (7, 8, 9, 10, 11, 12)
GROUP BY 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name -- Group by canonical country name

""")




In [15]:
df = regional_uniques.copy()
df['absolute_change'] = df['current_metric_avg'] - df['previous_metric_avg']

# Calculate Regional and Global Totals for the delta
regional_delta_total = df.groupby('wmf_region')['absolute_change'].sum().reset_index(name='total_region_delta')
global_delta_total = df['absolute_change'].sum()

# Merge the total regional delta back to the original DataFrame
df = pd.merge(df, regional_delta_total, on='wmf_region')

# Calculating Proportions for current metrics towards the regional and global totals
df['proportion_current_regional'] = (df['current_metric_avg'] / df.groupby('wmf_region')['current_metric_avg'].transform('sum')) * 100
df['proportion_current_global'] = (df['current_metric_avg'] / df['current_metric_avg'].sum()) * 100

# Update decomposition formula calculations to focus on country_code
df['formula_result_regional'] = abs((df['absolute_change'] * df['proportion_current_regional']) +
                                    ((df['proportion_current_regional'] - df.groupby('country_code')['proportion_current_regional'].transform('mean')) * (df['previous_metric_avg'] - df.groupby('wmf_region')['previous_metric_avg'].transform('sum'))))

df['formula_result_global'] = abs((df['absolute_change'] * df['proportion_current_global']) +
                                  ((df['proportion_current_global'] - df['proportion_current_global'].mean()) * (df['previous_metric_avg'] - df['previous_metric_avg'].sum())))

# Calculate the Total Sum of 'formula_result' per Region and Globally
df['formula_result_regional_sum'] = df.groupby('wmf_region')['formula_result_regional'].transform('sum')
df['total_formula_result_global'] = df['formula_result_global'].sum()

# Calculate percentages for formula results
df['formula_result_percentage_regional'] = (df['formula_result_regional'] / df['formula_result_regional_sum']) * 100
df['formula_result_percentage_global'] = (df['formula_result_global'] / df['total_formula_result_global']) * 100

# Prepare the final DataFrame for output with requested columns
df['month'] = 'Quarter' 

final_df = df[['month', 'wmf_region', 'country', 'current_metric_avg', 'previous_metric_avg', 'absolute_change', 
               'proportion_current_regional', 'formula_result_percentage_regional', 'proportion_current_global', 'formula_result_percentage_global']]

# Save the sorted DataFrame to a CSV file
print("Saving CSV file")
final_df.to_csv("unique_devices_time_series_analysis_quarterly.csv", index=False)

# Display the DataFrame
final_df

Saving CSV file


,month,wmf_region,country,current_metric_avg,previous_metric_avg,absolute_change,proportion_current_regional,formula_result_percentage_regional,proportion_current_global,formula_result_percentage_global
0,Quarter,Northern & Western Europe,Jersey,7.510967e+04,7.601200e+04,-9.023333e+02,0.018911,0.000003,0.004520,0.294562
1,Quarter,Northern & Western Europe,Luxembourg,8.461593e+05,7.003270e+05,1.458323e+05,0.213047,0.006001,0.050924,0.260316
2,Quarter,Northern & Western Europe,Åland,2.285800e+04,2.438800e+04,-1.530000e+03,0.005755,0.000002,0.001376,0.296885
3,Quarter,Northern & Western Europe,Faroe Islands,3.533133e+04,3.620733e+04,-8.760000e+02,0.008896,0.000002,0.002126,0.296331
4,Quarter,Northern & Western Europe,Isle of Man,7.124033e+04,7.337600e+04,-2.135667e+03,0.017937,0.000007,0.004287,0.294734
...,...,...,...,...,...,...,...,...,...,...
242,Quarter,South Asia,Nepal,1.918250e+06,2.063692e+06,-1.454413e+05,1.166321,0.062344,0.115444,0.212660
243,Quarter,South Asia,Maldives,1.782933e+05,1.382957e+05,3.999767e+04,0.108405,0.001594,0.010730,0.289981
244,Quarter,South Asia,Sri Lanka,2.293245e+06,2.254674e+06,3.857133e+04,1.394323,0.019766,0.138012,0.196063
245,Quarter,South Asia,India,1.427275e+08,1.396513e+08,3.076186e+06,86.780149,98.112304,8.589625,5.458610


## Mobile Wiki breakdown

In [8]:
# Query

regional_uniques_mobile = wmf.spark.run("""
SELECT 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name AS country, -- Use canonical country name from gdi
    AVG(CASE WHEN ud.month IN (7, 8, 9) THEN ud.uniques_estimate ELSE NULL END) AS previous_metric,
    AVG(CASE WHEN ud.month IN (10, 11, 12) THEN ud.uniques_estimate ELSE NULL END) AS current_metric
FROM 
    wmf.unique_devices_per_domain_monthly ud
JOIN 
    gdi.country_meta_data cmd ON ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.domain LIKE '%.m.wikipedia%'
    AND ud.year = 2023
GROUP BY 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name -- Group by canonical country name


""")


In [9]:

df = regional_uniques_mobile.copy()

# Calculate Absolute Change directly
df['absolute_change'] = df['current_metric'] - df['previous_metric']

# Calculate Regional and Global Totals
regional_totals = df.groupby('wmf_region')['current_metric', 'previous_metric'].sum().reset_index()
global_current_metric_total = df['current_metric'].sum()
global_previous_metric_total = df['previous_metric'].sum()

# Add Global Totals to df for proportion calculations
df['global_current_metric_total'] = global_current_metric_total
df['global_previous_metric_total'] = global_previous_metric_total

# Calculating Proportions and Formula Results
df['proportion_current_regional'] = df['current_metric'] / df.groupby('wmf_region')['current_metric'].transform('sum')
df['proportion_previous_regional'] = df['previous_metric'] / df.groupby('wmf_region')['previous_metric'].transform('sum')
df['proportion_current_global'] = df['current_metric'] / global_current_metric_total * 100
df['proportion_previous_global'] = df['previous_metric'] / global_previous_metric_total * 100

df['formula_result_regional'] = abs((df['absolute_change'] * df['proportion_current_regional']) +
                                    ((df['proportion_current_regional'] - df['proportion_previous_regional']) * (df['previous_metric'] - df.groupby('wmf_region')['previous_metric'].transform('sum'))))
df['formula_result_global'] = abs((df['absolute_change'] * df['proportion_current_global']) +
                                  ((df['proportion_current_global'] - df['proportion_previous_global']) * (df['previous_metric'] - global_previous_metric_total)))

# Calculate the Total Sum of 'formula_result' per Region and Globally
df['formula_result_regional_sum'] = df.groupby('wmf_region')['formula_result_regional'].transform('sum')
df['total_formula_result_global'] = df['formula_result_global'].sum()

# Calculate percentages
df['formula_result_percentage_regional'] = (df['formula_result_regional'] / df['formula_result_regional_sum']) * 100
df['formula_result_percentage_global'] = (df['formula_result_global'] / df['total_formula_result_global']) * 100

# Add ranking logic
df['rank_simple_calculation'] = df.groupby('wmf_region')['absolute_change'].rank(method='dense', ascending=False)
df['rank_formula_result'] = df.groupby('wmf_region')['formula_result_percentage_regional'].rank(method='dense', ascending=False)

# Sort the DataFrame by 'wmf_region', and 'absolute_change'
final_df = df.sort_values(by=['wmf_region', 'absolute_change'], ascending=[True, False])


final_df['month'] = 'Quarter'
final_df = final_df[['month', 'country', 'wmf_region',  'current_metric', 'previous_metric', 'absolute_change', 'proportion_current_regional', 'formula_result_percentage_regional', 'proportion_current_global', 'formula_result_percentage_global']]

# Saving the DataFrame to a CSV file
print("Saving CSV file")
final_df.to_csv("unique_devices_time_series_analysis_mobile_quarterly.csv", index=False, header = False)

# Display the DataFrame
final_df

Saving CSV file


/tmp/ipykernel_15263/3991077584.py:7: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  regional_totals = df.groupby('wmf_region')['current_metric', 'previous_metric'].sum().reset_index()


,month,country,wmf_region,current_metric,previous_metric,absolute_change,proportion_current_regional,formula_result_percentage_regional,proportion_current_global,formula_result_percentage_global
193,Quarter,Russia,Central & Eastern Europe & Central Asia,148108.334311,131777.449161,16330.885150,0.247516,9.303914,3.280490,5.173858
84,Quarter,Turkey,Central & Eastern Europe & Central Asia,72934.114594,65017.699308,7916.415286,0.121886,1.811691,1.615437,2.591173
216,Quarter,Kazakhstan,Central & Eastern Europe & Central Asia,24773.588813,17242.536706,7531.052106,0.041401,12.747506,0.548717,2.374486
213,Quarter,Ukraine,Central & Eastern Europe & Central Asia,56021.974585,49263.266535,6758.708050,0.093623,0.684912,1.240845,2.208173
124,Quarter,Uzbekistan,Central & Eastern Europe & Central Asia,13724.106557,8658.315519,5065.791038,0.022935,9.839807,0.303979,1.594688
...,...,...,...,...,...,...,...,...,...,...
171,Quarter,Mozambique,Sub-Saharan Africa,2054.989547,2244.308348,-189.318800,0.018558,3.019797,0.045517,0.051639
233,Quarter,Zambia,Sub-Saharan Africa,2093.376563,2299.493243,-206.116681,0.018905,3.216882,0.046367,0.056679
91,Quarter,Mauritius,Sub-Saharan Africa,1611.750419,1828.389098,-216.638679,0.014555,3.126864,0.035699,0.061463
79,Quarter,South Africa,Sub-Saharan Africa,18196.873031,20197.337688,-2000.464657,0.164333,21.653316,0.403047,0.552477


In [10]:
result = wmf.spark.run(""" select country_code, country from wmf.unique_devices_per_project_family_monthly where year = 2023 and month =  12""")
result

,country_code,country
0,HU,Hungary
1,BS,Bahamas
2,CZ,Czech Republic
3,BH,Bahrain
4,AU,Australia
...,...,...
2833,ST,São Tomé and Príncipe
2834,MF,Saint Martin
2835,BS,Bahamas
2836,MZ,Mozambique


In [11]:
result[result['country_code'] == 'CZ']

,country_code,country
2,CZ,Czech Republic
5,CZ,Czech Republic
61,CZ,Czech Republic
248,CZ,Czech Republic
276,CZ,Czech Republic
500,CZ,Czech Republic
732,CZ,Czech Republic
883,CZ,Czech Republic
963,CZ,Czech Republic
1072,CZ,Czech Republic
